# Predicción de precios de vivienda en California

Proyecto de Machine Learning supervisado sobre dos dominios distintos: predicción del precio medio de vivienda en California y predicción de abandono de clientes (churn) en banca. El desafío plantea dos preguntas centrales: *¿cuánto vale una vivienda dado su contexto?* y *¿qué clientes están en riesgo real de abandonar el banco?*

Este notebook recorre el pipeline completo — desde carga de datos y análisis exploratorio hasta feature engineering, entrenamiento, evaluación y diagnóstico de errores — para responder ambas preguntas con el estricto Ritual de los 7 pasos y una redacción honesta de los hallazgos: qué funciona, qué limita el modelo y cómo el propio RMSE nos mantiene humildes.


## Carga de datos

Trabajamos con dos datasets reales en formato CSV ubicados en `data/raw/`:

- **California Housing Prices**: precios de vivienda en California (1990) a nivel de bloque censal, con el target `median_house_value` en **dólares reales**.
- **Bank Customer Churn**: 10.000 clientes de un banco europeo con su indicador de abandono (`Exited`).

Ambos se cargan tal cual, sin modificaciones sobre los archivos originales.


In [ ]:
#  Imports y Configuración 

# Importamos las librerías de análisis y visualización de datos.
import pandas as pd          
import numpy as np           
import matplotlib.pyplot as plt  
import seaborn as sns        

# 1. División de Datos y Optimización (sklearn.model_selection) 
from sklearn.model_selection import train_test_split # train_test_split: division de datos en train y test (el modelo aprende con train y se mide con test, como en el reto).

# 2. Limpieza y Transformación de Datos  (sklearn.preprocessing e impute) 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
#   StandardScaler: escala los números para que tengan media 0 y desvía estándar 1.
#     Crucial para Regresión Logística o K-Means: evita que una variable con números
#     gigantes (salario) opaque a una con números pequeños (edad).
#   OneHotEncoder: convierte texto/categorías (“Rojo”, “Verde”) en columnas de 0/1.
#     Los modelos sólo entienden números, así que este paso es obligatorio.
from sklearn.compose import ColumnTransformer
#   ColumnTransformer: aplica transformaciones DIFERENTES a columnas DIFERENTES en un
#     solo paso (ej: StandardScaler a numéricas + OneHotEncoder a texto a la vez).
from sklearn.pipeline import Pipeline
#   Pipeline: “fábrica” o cadena de montaje. Pega preprocesamiento + modelo en un solo
#     objeto; con .fit() limpia, transforma y entrena automáticamente y en orden.
from sklearn.impute import SimpleImputer
#   SimpleImputer: llena los datos faltantes (nulos/NaN) con el promedio, la mediana
#     o el valor más repetido de la columna (usamos mediana para total_bedrooms).
from sklearn.cluster import KMeans
#   KMeans: aprendizaje NO supervisado. Agrupa datos por similitud sin etiquetas
#     previas (creamos ZONAS GEOGRÁFICAS a partir de latitud/longitud).


# 3. Modelos de Machine Learning (Algoritmos)
from sklearn.linear_model import LinearRegression, LogisticRegression
#   LinearRegression: regresión clásica. Predice un valor continuo (precio de una casa)
#     dibujando la línea que mejor se ajusta (mínimos cuadrados / OLS).
#   LogisticRegression: para CLASIFICACIÓN binaria (predecir si ocurre un evento o no,
#     como el Churn o el Spam). Devuelve la probabilidad de pertenecer a la clase 1.


# 4. Metricas de Evaluacion de Modelos (sklearn.metrics)
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, classification_report,
                             confusion_matrix)
from sklearn.metrics import ConfusionMatrixDisplay
#   --- Para REGRESIÓN (predicción de números) ---
#   mean_squared_error: calcula el MSE (error cuadrático medio). Con raíz cuadrada da
#     el RMSE, el error promedio en las unidades originales ($).
#   r2_score (R²): qué porcentaje de la variación de los datos explica el modelo
#     (0 a 1, donde 1 = ajuste perfecto). Lo usamos para California.
#   --- Para CLASIFICACIÓN (predicción de categorías) ---
#   accuracy_score: % total de predicciones correctas (a cuántos acerté en total).
#   precision_score: de los clasificados como “Positivos”, cuántos lo eran realmente
#     (de los que predije que harían Churn, cuántos sí lo hicieron). Evita falsos positivos.
#   recall_score: sensibilidad. De los que realmente hicieron Churn, cuántos detectó.
#     Evita falsos negativos.
#   classification_report: resumen completo (accuracy, precision, recall, f1 por clase).
#   confusion_matrix: cruza valores reales vs predicciones (aciertos, falsos pos./neg.).
#   ConfusionMatrixDisplay: grafica la matriz de confusión con colores para leerla fácil.


# c) Configuracion general
import warnings
warnings.filterwarnings('ignore')          
sns.set_theme(style='whitegrid')          

SEED = 42                                  # semilla fija: misma partición en cada ejecución (reproducibilidad)

# Rutas relativas al notebook (data/raw/) y carga de ambos datasets
# Churn: dataset clásico de 10.000 clientes (abbas829/bank-customer-churn -> Bank_Churn.csv).
CAL = r'data/raw/housing.csv'              # dataset de regresión (California)
BNK = r'data/raw/Bank_Churn.csv'           # dataset de clasificación (Bank churn)

df_cal_raw = pd.read_csv(CAL)             
df_bnk_raw = pd.read_csv(BNK)              

print('California Housing:', df_cal_raw.shape[0], 'filas x', df_cal_raw.shape[1], 'columnas')
print('Bank Churn      :', df_bnk_raw.shape[0], 'filas x', df_bnk_raw.shape[1], 'columnas')
